# Chapter 3: The radius of gyration and the ratio X/R_g

*Code cells: MIT licence (`LICENSE`). Text and figures: CC BY 4.0 (`DATA_LICENSE.md`). Part of the code-and-data repository for "Mean span dimension of an ideal wormlike chain" (Y. Wang, 2026).*

In [1]:
import sys, json, math, time
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'tutorial' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / 'code'))
import numpy as np, pandas as pd, sympy as sp
import matplotlib.pyplot as plt
pd.set_option('display.precision', 8)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 9})
REF = pd.read_csv(ROOT / 'data' / 'wlc_mean_span_reference.csv')
LAM, F = REF['lambda'].to_numpy(), REF['F_continuum'].to_numpy()
Z_M, B_COEF, A_GAUSS = 0.71821164, 0.64680, 4/math.sqrt(3*math.pi)
def check(name, value, expected, tol):
    ok = abs(value - expected) <= tol
    print(('PASS' if ok else 'FAIL') + f'  {name}: {value:.10g}  (expected {expected:.10g}, tol {tol:g})')
    return ok

## Derivation

$R_g^2=\big\langle L^{-1}\!\int_0^L|\mathbf r(s)-\mathbf r_{\rm cm}|^2ds\big\rangle
=(2L^2)^{-1}\!\int_0^L\!\!\int_0^L\langle|\mathbf r(s)-\mathbf r(s')|^2\rangle\,ds\,ds'$,
and the mean-square separation of two contour points at distance $h$ is $2p^2(h/p-1+e^{-h/p})$.
SymPy performs the double integral:

In [2]:
lam, u = sp.symbols('lambda u', positive=True)
msd = 2*(u-1+sp.exp(-u))                                  # <|r(s)-r(s')|^2>/p^2 with u=|s-s'|/p
g = sp.simplify(1/lam**2 * sp.integrate((lam-u)*msd, (u, 0, lam)))   # (2L^2)^-1 * 2 * int_0^L (L-h) msd dh  (the 2 counts s<s' and s>s')
g_article = lam/3 - 1 + 2/lam - 2*(1-sp.exp(-lam))/lam**2
print('g(lambda) =', g)
print('matches the radius-of-gyration formula:', sp.simplify(g - g_article) == 0)
print('series   :', sp.series(g, lam, 0, 6))
print('limits   : rod R_g^2 -> lambda^2/12 ;  coil R_g^2 -> lambda/3')

g(lambda) = lambda/3 - 1 + 2/lambda - 2/lambda**2 + 2*exp(-lambda)/lambda**2
matches the radius-of-gyration formula: True
series   : lambda**2/12 - lambda**3/60 + lambda**4/360 - lambda**5/2520 + O(lambda**6)
limits   : rod R_g^2 -> lambda^2/12 ;  coil R_g^2 -> lambda/3


## The ratio and its limits

$X/R_g=F/\sqrt g$ tends to $\sqrt3$ for a rod ($F\to\lambda/2$, $g\to\lambda^2/12$) and to $4/\sqrt\pi$ for a
coil ($F\to4\sqrt{\lambda/3\pi}$, $g\to\lambda/3$). With the short-chain span $F=\lambda/2-37\lambda^2/768$ (Chapter 4):

In [3]:
Fshort = lam/2 - sp.Rational(37,768)*lam**2
gser = sp.series(g/lam**2, lam, 0, 3).removeO()                  # g = lambda^2 (1/12 - lambda/60 + ...)
ratio_short = sp.series(Fshort/(lam*sp.sqrt(gser)), lam, 0, 2).removeO()
print('X/Rg (short) / sqrt(3) =', sp.simplify(ratio_short/sp.sqrt(3)), '   coefficient 1/10 - 37/384 =', sp.Rational(1,10)-sp.Rational(37,384))
zM, Bc = sp.symbols('z_M B')
Flong = 4*sp.sqrt(lam/(3*sp.pi)) - 2*zM + Bc/sp.sqrt(lam)
g_long = lam/3 - 1 + 2/lam                                        # exponentially small terms dropped
v = sp.symbols('v', positive=True)                                # v = 1/sqrt(lambda)
expr = (Flong/sp.sqrt(g_long)).subs(lam, 1/v**2)
print('X/Rg (long)  =', sp.series(expr, v, 0, 3).removeO())

X/Rg (short) / sqrt(3) = 7*lambda/1920 + 1    coefficient 1/10 - 37/384 = 7/1920
X/Rg (long)  = v**2*(sqrt(3)*B + 6/sqrt(pi)) - 2*sqrt(3)*v*z_M + 4/sqrt(pi)


The long-chain coefficients are $-2\sqrt3\,z_M$ at order $\lambda^{-1/2}$ and $\sqrt3B+6/\sqrt\pi$ at order
$\lambda^{-1}$, as in the long-chain expansion of X/R_g. **Status:** the limits are exact; the short-chain slope $7/1920$ is
derived; the long-chain terms are conditional because they inherit $z_M$ and $B$.

## Numbers quoted in the article

In [4]:
import plot_main_figures as pmf
ratio = F/np.sqrt(pmf.rg2_over_p2(LAM))
for x in (1, 10, 100, 1000, 5000, 1e5):
    print(f'lambda={x:>7g}   X/Rg = {float(ratio[LAM==x][0]):.4f}')
coil = 4/math.sqrt(math.pi)
print('below coil value at 5000 : %.2f %%' % (100*(1-float(ratio[LAM==5000][0])/coil)))
print('below coil value at 1e5  : %.2f %%' % (100*(1-float(ratio[LAM==1e5][0])/coil)))
lam1 = (2*math.sqrt(3)*Z_M/(0.01*coil))**2
print('1 %% approach (leading term):  lambda ~ %.3g' % lam1)
check('X/Rg at 5000', float(ratio[LAM==5000][0]), 2.2225, 5e-5)

lambda=      1   X/Rg = 1.7382
lambda=     10   X/Rg = 1.8207
lambda=    100   X/Rg = 2.0495
lambda=   1000   X/Rg = 2.1825
lambda=   5000   X/Rg = 2.2225
lambda= 100000   X/Rg = 2.2489
below coil value at 5000 : 1.52 %
below coil value at 1e5  : 0.35 %
1 % approach (leading term):  lambda ~ 1.22e+04
PASS  X/Rg at 5000: 2.222463923  (expected 2.2225, tol 5e-05)


True

## The identity $\delta=X/2$, step by step

Take a hard wall at height zero with normal $\hat{\mathbf e}$, and dilute chains that the wall affects only
by removing the conformations that cross it. Fix a reference point $s$ on the contour and place it at
height $h$.

1. The conformation survives if its lowest point is above the wall, i.e. if $Z(s)-\min Z<h$, where
   $Z=\mathbf r\cdot\hat{\mathbf e}$. So $c_s(h)/c_{\rm b}=\Pr\{Z(s)-\min Z<h\}$.
2. The depletion thickness is $\delta=\int_0^\infty[1-c_s(h)/c_{\rm b}]\,dh=\int_0^\infty\Pr\{Z(s)-\min Z\ge h\}\,dh
   =\langle Z(s)-\min Z\rangle$, the mean of a nonnegative variable.
3. A rotation by $\pi$ about an axis in the wall plane maps $Z\to-Z$ and leaves the ensemble of
   unoriented chains unchanged, so $\langle Z(s)-\min Z\rangle=\langle\max Z-Z(s)\rangle$.
4. The two sum to $\langle\max Z-\min Z\rangle=X$. Hence $\delta=X/2$ for any reference point $s$.

Nothing in the argument is specific to the wormlike chain. The cell tests it on sampled chains at
$\lambda=1$ for four reference points; the two one-sided averages differ from point to point within sampling
error only.

In [5]:
from sphere_heat_kernel import heat_inverse, heat_step
def projected_paths(lam, h=0.01, N=40000, seed=3):
    u_, cdf, _ = heat_inverse(h)
    rng = np.random.default_rng(seed)
    t = rng.normal(size=(N, 3)); t /= np.linalg.norm(t, axis=1)[:, None]
    n = int(round(lam/h)); Zp = np.zeros((n+1, N))
    for k in range(n):
        tn = heat_step(t, u_, cdf, rng)
        Zp[k+1] = Zp[k] + 0.5*(t[:, 2] + tn[:, 2])*h; t = tn
    return Zp
Zp = projected_paths(1.0); lo, hi = Zp.min(axis=0), Zp.max(axis=0); N = Zp.shape[1]
F1 = float(F[LAM == 1.0][0])
print(f'X/(2p) from the reference table: {F1/2:.4f};  from these chains: {(hi-lo).mean()/2:.4f} +/- {(hi-lo).std()/2/math.sqrt(N):.4f}')
for frac, name in ((0, 'end'), (0.25, 'quarter point'), (0.5, 'midpoint'), (1.0, 'other end')):
    k = int(frac*(Zp.shape[0]-1)); d = Zp[k] - lo
    print(f'reference point: {name:13s}  <Z(s) - min Z> = {d.mean():.4f} +/- {d.std()/math.sqrt(N):.4f}    <max Z - Z(s)> = {(hi-Zp[k]).mean():.4f}')
k = Zp.shape[0]//2; d = Zp[k] - lo; hh = np.linspace(0, 1.0, 6)
print('\nconcentration profile of the midpoint, c(h)/c_b = Pr{Z(s) - min Z < h}:')
print('   h/p     :', np.round(hh, 2)); print('   c/c_b   :', np.round([(d < x).mean() for x in hh], 3))

X/(2p) from the reference table: 0.2285;  from these chains: 0.2277 +/- 0.0006
reference point: end            <Z(s) - min Z> = 0.2269 +/- 0.0014    <max Z - Z(s)> = 0.2285
reference point: quarter point  <Z(s) - min Z> = 0.2274 +/- 0.0009    <max Z - Z(s)> = 0.2280
reference point: midpoint       <Z(s) - min Z> = 0.2278 +/- 0.0007    <max Z - Z(s)> = 0.2277
reference point: other end      <Z(s) - min Z> = 0.2282 +/- 0.0014    <max Z - Z(s)> = 0.2272

concentration profile of the midpoint, c(h)/c_b = Pr{Z(s) - min Z < h}:
   h/p     : [0.  0.2 0.4 0.6 0.8 1. ]
   c/c_b   : [0.    0.438 0.869 1.    1.    1.   ]


The same argument gives the partition coefficient of a slit of width $H$ wider than the largest span:
a conformation fits if $X_c<H$, and the accessible heights of its lowest point have total length $H-X_c$,
so $K=\langle H-X_c\rangle/H=1-X/H$ (here $H\ge L$ suffices, since $X_c\le L$).

## The depletion layer at a wall

By the identity of Wang, Hansen, Peters and Hassager (J. Chem. Phys. 129, 074904, 2008) the depletion thickness of
any reference point at a hard wall is $\delta=X/2$. For a Gaussian chain $\delta=2R_g/\sqrt\pi$. With the long-chain
expansions of $F$ and $g$, the wall offset $2R_g/\sqrt\pi-\delta$ tends to $z_Mp$:
$$\frac{2R_g}{\sqrt\pi}-\delta=z_Mp-\Big(\frac B2+\sqrt{\frac3\pi}\Big)\frac{p}{\sqrt\lambda}+\dots,$$
so the approach is slow. The cell computes the offset from the spectral table (Fig. 3d).

In [6]:
g = pmf.rg2_over_p2(LAM)
offset = 2*np.sqrt(g)/math.sqrt(math.pi) - F/2
for x in (10, 100, 1000, 1e4, 1e5):
    o = float(offset[LAM==x][0])
    print(f'lambda={x:>7g}   offset/p = {o:.4f}   asymptotic form = {Z_M-(B_COEF/2+math.sqrt(3/math.pi))/math.sqrt(x):.4f}')
print('limit z_M =', Z_M)
print('short chains: (L/4 - delta)/(L^2/p) at 0.01 =', (0.01/4-float(F[LAM==0.01][0])/2)/1e-4, ' (37/1536 =', 37/1536, ')')

lambda=     10   offset/p = 0.3457   asymptotic form = 0.3069
lambda=    100   offset/p = 0.5894   asymptotic form = 0.5882
lambda=   1000   offset/p = 0.6771   asymptotic form = 0.6771
lambda=  10000   offset/p = 0.7052   asymptotic form = 0.7052
lambda= 100000   offset/p = 0.7141   asymptotic form = 0.7141
limit z_M = 0.71821164
short chains: (L/4 - delta)/(L^2/p) at 0.01 = 0.024095238638000872  (37/1536 = 0.024088541666666668 )
